# V18-v2 PlasmaB/B Cell Antibody Production Investigation

**Core hypothesis:** IT-IA-CR share common PlasmaB/B cell changes absent in AR,
potentially reflecting impaired anti-HBs antibody production capacity.

**Strategy:**
1. Test antibody-production-related genes in PlasmaB and B cells
2. Find genes with pattern: (NL→IT sig OR NL→IA sig OR NL→CR sig) AND NL→AR NS
3. Specifically: PRDM1, JCHAIN, XBP1, IRF4, AICDA in PlasmaB/B
4. Broad scan: ALL genes in PlasmaB/B for IT-IA-CR common pattern
5. Compare with known IT-persistent PlasmaB cytotoxic program (TYROBP/FCER1G/GZMB)

**Key question:** Is there evidence that PlasmaB cells are redirected from
antibody secretion toward a cytotoxic program during chronic infection?

In [1]:
# ============================================================
# CELL 0: GPU SETUP & VERIFICATION
# ============================================================
!pip install scanpy anndata matplotlib seaborn scipy -q

import subprocess
import sys

print("=" * 70)
print("  V18 C3: GPU ENVIRONMENT SETUP")
print("=" * 70)

# GPU detection
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                            '--format=csv,noheader'], capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"✅ GPU detected: {gpu_info}")
except:
    print("⚠️ No GPU detected — will use CPU fallback")

# Install CuPy for GPU acceleration
try:
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} ready")
    print(f"   GPU memory: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB total")
except ImportError:
    print("📦 Installing CuPy...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cupy-cuda12x', '-q'])
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 139.7 MB/s eta 0:00:00
  V18 C3: GPU ENVIRONMENT SETUP
✅ GPU detected: NVIDIA A100-SXM4-80GB, 81920 MiB, 8.0
✅ CuPy 14.0.1 ready
   GPU memory: 85.1 GB total


In [2]:
# Cell 1: Setup
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import os, time, warnings
warnings.filterwarnings('ignore')

try:
    _ = adata.shape
    print(f'adata loaded: {adata.shape}')
except:
    from google.colab import drive
    drive.mount('/content/drive')
    import scanpy as sc
    DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
    print('Loading h5ad...')
    adata = sc.read_h5ad(DATA_PATH)
    print(f'Loaded: {adata.shape}')

RESULTS_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2'
SAVE_DIR = os.path.join(RESULTS_DIR, 'PlasmaB_antibody_investigation_26Mar14')
os.makedirs(SAVE_DIR, exist_ok=True)

obs = adata.obs.copy()

# Column detection
COL_STAGE = 'Stage' if 'Stage' in obs.columns else [c for c in obs.columns if 'stage' in c.lower()][0]
COL_LINEAGE = 'major_lineage' if 'major_lineage' in obs.columns else [c for c in obs.columns if 'lineage' in c.lower()][0]
if 'tissue' in obs.columns:
    COL_TISSUE = 'tissue'
elif 'Tissue' in obs.columns:
    COL_TISSUE = 'Tissue'
else:
    for c in ['sample', 'Sample', 'orig.ident']:
        if c in obs.columns:
            obs['tissue_derived'] = obs[c].apply(
                lambda x: 'Liver' if ('Liver' in str(x) or '_L_' in str(x) or str(x).endswith('_L'))
                else ('Blood' if ('PBMC' in str(x) or '_P_' in str(x) or str(x).endswith('_P') or 'Blood' in str(x))
                else 'Unknown'))
            COL_TISSUE = 'tissue_derived'
            break
COL_DONOR = None
for c in ['donor', 'Donor', 'patient', 'subject', 'donor_id']:
    if c in obs.columns:
        COL_DONOR = c
        break
if COL_DONOR is None:
    for c in ['sample', 'Sample', 'orig.ident']:
        if c in obs.columns:
            obs['donor_derived'] = obs[c].astype(str).str.split('_').str[1]
            COL_DONOR = 'donor_derived'
            break

print(f'Columns: Stage={COL_STAGE}, Lineage={COL_LINEAGE}, Tissue={COL_TISSUE}, Donor={COL_DONOR}')
print(f'\nPlasmaB cells: {(obs[COL_LINEAGE] == "PlasmaB").sum()}')
print(f'B cells: {(obs[COL_LINEAGE] == "B").sum()}')
print(f'\nPlasmaB per stage-tissue:')
print(obs[obs[COL_LINEAGE] == 'PlasmaB'].groupby([COL_STAGE, COL_TISSUE]).size())

Mounted at /content/drive
Loading h5ad...
Loaded: (243000, 24452)
Columns: Stage=Stage, Lineage=major_lineage, Tissue=tissue, Donor=donor_derived

PlasmaB cells: 2083
B cells: 21325

PlasmaB per stage-tissue:
Stage  tissue
CR     Blood     259
       Liver     107
AR     Blood     229
       Liver      63
IA     Blood     213
       Liver     145
IT     Blood     197
       Liver     168
NL     Blood      68
       Liver     634
dtype: int64


In [3]:
# Cell 2: Check which antibody-related genes exist in dataset

# Priority: antibody production / plasma cell function
ab_genes = [
    # Plasma cell master regulators
    'PRDM1',    # BLIMP-1: master TF for plasma cell differentiation + Ab secretion
    'IRF4',     # cooperates with PRDM1 for plasma cell commitment
    'XBP1',     # UPR, essential for Ab mass secretion

    # Antibody production machinery
    'JCHAIN',   # J-chain: IgA/IgM polymerization, secretory Ab
    'AICDA',    # AID: class switch recombination (IgM→IgG)
    'MZB1',     # ER chaperone for Ab assembly
    'SDC1',     # CD138: plasma cell marker
    'CD38',     # plasma cell marker
    'TNFRSF17', # BCMA: plasma cell survival
    'CD27',     # memory B / pre-plasma
    'CD19',     # B cell identity

    # B cell activation / GC
    'BCL6',     # germinal center TF
    'BCL2',     # survival
    'PAX5',     # B cell identity TF (lost in plasma cells)
    'IL2RA',    # CD25: B activation
    'CD69',     # activation

    # Already known IT-persistent in PlasmaB (cytotoxic program)
    'TYROBP',   # NK adaptor in PlasmaB
    'FCER1G',   # NK adaptor in PlasmaB
    'GZMB',     # granzyme B in PlasmaB
    'GNLY',     # granulysin

    # Epigenetic (PlasmaB DNMT1 already known)
    'DNMT1', 'DNMT3A', 'TET2',

    # Signaling
    'JAK1', 'MTOR',

    # Additional B/PlasmaB relevant
    'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4',  # IgG subclass heavy chains
    'IGHA1', 'IGHA2',  # IgA heavy chains
    'IGHM',    # IgM heavy chain
    'IGHD',    # IgD
    'IGKC', 'IGLC1', 'IGLC2',  # light chains
]

all_genes = list(adata.var_names)
found = [g for g in ab_genes if g in all_genes]
missing = [g for g in ab_genes if g not in all_genes]

print(f'Found: {len(found)}/{len(ab_genes)}')
print(f'Found: {found}')
print(f'\nMissing: {missing}')

# Also check for immunoglobulin genes broadly
ig_genes = [g for g in all_genes if g.startswith('IGH') or g.startswith('IGK') or g.startswith('IGL')]
print(f'\nAll Ig genes in dataset ({len(ig_genes)}): {ig_genes}')

Found: 35/36
Found: ['PRDM1', 'IRF4', 'XBP1', 'JCHAIN', 'AICDA', 'MZB1', 'SDC1', 'CD38', 'TNFRSF17', 'CD27', 'CD19', 'BCL6', 'BCL2', 'PAX5', 'IL2RA', 'CD69', 'TYROBP', 'FCER1G', 'GZMB', 'GNLY', 'DNMT1', 'DNMT3A', 'TET2', 'JAK1', 'MTOR', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4', 'IGHA1', 'IGHA2', 'IGHM', 'IGHD', 'IGKC', 'IGLC2']

Missing: ['IGLC1']

All Ig genes in dataset (230): ['IGKV1OR1-1', 'IGKV3OR2-268', 'IGKC', 'IGKJ5', 'IGKJ1', 'IGKV4-1', 'IGKV5-2', 'IGKV7-3', 'IGKV2-4', 'IGKV1-5', 'IGKV1-6', 'IGKV3-7', 'IGKV1-8', 'IGKV1-9', 'IGKV2-10', 'IGKV3-11', 'IGKV1-12', 'IGKV1-13', 'IGKV3-15', 'IGKV1-16', 'IGKV1-17', 'IGKV2-18', 'IGKV3-20', 'IGKV6-21', 'IGKV2-24', 'IGKV2-26', 'IGKV1-27', 'IGKV2-28', 'IGKV2-29', 'IGKV2-30', 'IGKV1-33', 'IGKV1-37', 'IGKV1-39', 'IGKV2-40', 'IGKV2D-40', 'IGKV1D-39', 'IGKV1D-37', 'IGKV1D-33', 'IGKV2D-30', 'IGKV2D-29', 'IGKV2D-28', 'IGKV1D-27', 'IGKV2D-26', 'IGKV2D-24', 'IGKV6D-21', 'IGKV3D-20', 'IGKV2D-18', 'IGKV6D-41', 'IGKV1D-17', 'IGKV1D-16', 'IGKV3D-15', 'IGKV1D

In [4]:
# Cell 3: Pre-extract all found genes
t0 = time.time()

gene_expr = {}
for gene in found:
    try:
        col = adata[:, gene].X
        if hasattr(col, 'toarray'):
            col = col.toarray().flatten()
        else:
            col = np.asarray(col).flatten()
        gene_expr[gene] = col
    except Exception as e:
        print(f'  Error {gene}: {e}')

# Also extract any Ig genes found
for gene in ig_genes:
    if gene not in gene_expr:
        try:
            col = adata[:, gene].X
            if hasattr(col, 'toarray'):
                col = col.toarray().flatten()
            else:
                col = np.asarray(col).flatten()
            gene_expr[gene] = col
        except:
            pass

print(f'Pre-extracted {len(gene_expr)} genes in {time.time()-t0:.1f}s')

Pre-extracted 255 genes in 79.1s


In [5]:
# Cell 4: Fast donor-level test

def fast_donor_test(gene, lineage, tissue, group_a, group_b):
    if gene not in gene_expr:
        return None
    mask = (
        (obs[COL_STAGE].isin([group_a, group_b])) &
        (obs[COL_LINEAGE] == lineage) &
        (obs[COL_TISSUE] == tissue)
    )
    if mask.sum() == 0:
        return None
    cell_indices = np.where(mask.values)[0]
    expr = gene_expr[gene][cell_indices]
    temp = obs.loc[mask, [COL_DONOR, COL_STAGE]].copy()
    temp['expr'] = expr
    donor_means = temp.groupby([COL_DONOR, COL_STAGE])['expr'].mean().reset_index()
    vals_a = donor_means[donor_means[COL_STAGE] == group_a]['expr'].dropna().values
    vals_b = donor_means[donor_means[COL_STAGE] == group_b]['expr'].dropna().values
    n_a, n_b = len(vals_a), len(vals_b)
    if n_a < 2 or n_b < 2:
        return None
    mean_a, mean_b = float(np.mean(vals_a)), float(np.mean(vals_b))
    try:
        _, p_val = mannwhitneyu(vals_a, vals_b, alternative='two-sided')
        p_val = float(p_val)
    except:
        p_val = 1.0
    if mean_a > 1e-10:
        pct = (mean_b - mean_a) / mean_a * 100
    elif mean_b > 1e-10:
        pct = float('inf')
    else:
        pct = 0.0
    from_zero = (mean_a < 1e-10 and mean_b > 1e-10)
    n_consistent = sum(1 for va in vals_a for vb in vals_b
                       if (mean_b > mean_a and vb > va) or (mean_b <= mean_a and vb < va))
    return {
        'p': round(p_val, 4), 'pct': round(pct, 1),
        'dir': '+' if mean_b > mean_a else '-',
        'sig': '*' if p_val < 0.05 else ('(t)' if p_val < 0.10 else ''),
        'cons': f'{n_consistent}/{n_a * n_b}',
        'mean_a': round(mean_a, 4), 'mean_b': round(mean_b, 4),
        'n_a': n_a, 'n_b': n_b, 'from_zero': from_zero,
    }

def fmt(res):
    if res is None:
        return 'ND'
    if res['from_zero']:
        return f'{res["sig"]}from_zero p={res["p"]:.3f}'
    return f'{res["sig"]}{res["dir"]}{abs(res["pct"]):.0f}% p={res["p"]:.3f}'

print('Functions defined')

Functions defined


In [6]:
# Cell 5: PRIORITY — Antibody production genes in PlasmaB and B
# 4 comparisons: NL→IT, NL→IA, NL→CR, NL→AR
# Looking for: chronic common pattern (IT/IA/CR sig, AR NS)

priority_genes = ['PRDM1', 'IRF4', 'XBP1', 'JCHAIN', 'AICDA',
                  'MZB1', 'SDC1', 'CD38', 'TNFRSF17', 'CD27',
                  'BCL6', 'PAX5', 'IL2RA',
                  'TYROBP', 'FCER1G', 'GZMB', 'GNLY',
                  'DNMT1', 'DNMT3A', 'TET2', 'JAK1', 'MTOR']
priority_genes = [g for g in priority_genes if g in gene_expr]

lineages_bc = ['PlasmaB', 'B']
tissues = ['Liver', 'Blood']

print('='*150)
print('ANTIBODY PRODUCTION GENES IN PlasmaB/B: Full Disease Spectrum')
print('='*150)
print(f'{"Gene":>10s} | {"Lineage":>8s} | {"Tissue":>6s} | '
      f'{"NL->IT":>18s} | {"NL->IA":>18s} | {"NL->CR":>18s} | '
      f'{"NL->AR":>18s} | {"Chronic?":>10s}')
print('-'*150)

chronic_common = []  # IT/IA/CR at least 2 sig, AR NS

for gene in priority_genes:
    for lin in lineages_bc:
        for tis in tissues:
            nl_it = fast_donor_test(gene, lin, tis, 'NL', 'IT')
            nl_ia = fast_donor_test(gene, lin, tis, 'NL', 'IA')
            nl_cr = fast_donor_test(gene, lin, tis, 'NL', 'CR')
            nl_ar = fast_donor_test(gene, lin, tis, 'NL', 'AR')

            # Count how many chronic stages are significant
            it_sig = nl_it and nl_it['p'] < 0.05
            ia_sig = nl_ia and nl_ia['p'] < 0.05
            cr_sig = nl_cr and nl_cr['p'] < 0.05
            ar_sig = nl_ar and nl_ar['p'] < 0.05

            chronic_count = sum([it_sig, ia_sig, cr_sig])
            any_sig = chronic_count > 0 or ar_sig

            if not any_sig:
                continue

            # Chronic common: >=2 chronic stages sig, AR NS
            is_chronic = chronic_count >= 2 and not ar_sig
            # Also flag: even 1 chronic sig + AR NS is interesting
            is_partial = chronic_count >= 1 and not ar_sig

            label = ''
            if is_chronic:
                label = '★CHRONIC'
            elif is_partial:
                label = '(partial)'
            elif ar_sig and chronic_count == 0:
                label = 'AR-only'

            if is_chronic:
                chronic_common.append({
                    'Gene': gene, 'Lineage': lin, 'Tissue': tis,
                    'NL_IT': fmt(nl_it), 'NL_IA': fmt(nl_ia),
                    'NL_CR': fmt(nl_cr), 'NL_AR': fmt(nl_ar),
                    'IT_sig': it_sig, 'IA_sig': ia_sig, 'CR_sig': cr_sig,
                })

            print(f'{gene:>10s} | {lin:>8s} | {tis:>6s} | '
                  f'{fmt(nl_it):>18s} | {fmt(nl_ia):>18s} | '
                  f'{fmt(nl_cr):>18s} | {fmt(nl_ar):>18s} | '
                  f'{label:>10s}')

print(f'\n=== CHRONIC COMMON (>=2 chronic stages sig, AR NS): {len(chronic_common)} ===')
for r in chronic_common:
    print(f'  {r["Gene"]:>10s} {r["Lineage"]:>8s} {r["Tissue"]:>6s}: '
          f'IT={"*" if r["IT_sig"] else " "} IA={"*" if r["IA_sig"] else " "} '
          f'CR={"*" if r["CR_sig"] else " "}')

ANTIBODY PRODUCTION GENES IN PlasmaB/B: Full Disease Spectrum
      Gene |  Lineage | Tissue |             NL->IT |             NL->IA |             NL->CR |             NL->AR |   Chronic?
------------------------------------------------------------------------------------------------------------------------------------------------------
      IRF4 |  PlasmaB |  Liver |     *+150% p=0.030 |     *+148% p=0.030 |       +75% p=0.714 |       +94% p=0.167 |   ★CHRONIC
    JCHAIN |  PlasmaB |  Liver |      *-27% p=0.017 |    (t)-25% p=0.082 |       -15% p=0.381 |        -6% p=0.714 |  (partial)
    JCHAIN |        B |  Liver |    (t)+56% p=0.082 |       +50% p=0.126 |     *+111% p=0.048 |     *+169% p=0.024 |           
    JCHAIN |        B |  Blood |     *+111% p=0.016 |    (t)+90% p=0.064 |      +107% p=0.143 |     *+156% p=0.036 |           
     AICDA |        B |  Liver | *from_zero p=0.048 |  from_zero p=0.361 |        -0% p=1.000 | (t)from_zero p=0.052 |  (partial)
     AICDA |     

In [7]:
# Cell 6: Immunoglobulin genes in PlasmaB/B
# If IgG heavy chain genes are suppressed in IT-IA-CR but normal in AR,
# that would be direct evidence of impaired antibody production

ig_in_expr = [g for g in gene_expr if g.startswith('IGH') or g.startswith('IGK') or g.startswith('IGL')]
print(f'Ig genes available: {len(ig_in_expr)}')
print(f'{ig_in_expr}\n')

if ig_in_expr:
    print('='*150)
    print('IMMUNOGLOBULIN GENES IN PlasmaB/B')
    print('='*150)
    print(f'{"Gene":>10s} | {"Lineage":>8s} | {"Tissue":>6s} | '
          f'{"NL->IT":>18s} | {"NL->IA":>18s} | {"NL->CR":>18s} | '
          f'{"NL->AR":>18s} | {"Note":>10s}')
    print('-'*150)

    for gene in ig_in_expr:
        for lin in lineages_bc:
            for tis in tissues:
                nl_it = fast_donor_test(gene, lin, tis, 'NL', 'IT')
                nl_ia = fast_donor_test(gene, lin, tis, 'NL', 'IA')
                nl_cr = fast_donor_test(gene, lin, tis, 'NL', 'CR')
                nl_ar = fast_donor_test(gene, lin, tis, 'NL', 'AR')

                any_sig = any(r and r['p'] < 0.10 for r in [nl_it, nl_ia, nl_cr, nl_ar])
                if not any_sig:
                    continue

                print(f'{gene:>10s} | {lin:>8s} | {tis:>6s} | '
                      f'{fmt(nl_it):>18s} | {fmt(nl_ia):>18s} | '
                      f'{fmt(nl_cr):>18s} | {fmt(nl_ar):>18s} |')
else:
    print('No Ig genes found in dataset')

Ig genes available: 230
['IGHG1', 'IGHG2', 'IGHG3', 'IGHG4', 'IGHA1', 'IGHA2', 'IGHM', 'IGHD', 'IGKC', 'IGLC2', 'IGKV1OR1-1', 'IGKV3OR2-268', 'IGKJ5', 'IGKJ1', 'IGKV4-1', 'IGKV5-2', 'IGKV7-3', 'IGKV2-4', 'IGKV1-5', 'IGKV1-6', 'IGKV3-7', 'IGKV1-8', 'IGKV1-9', 'IGKV2-10', 'IGKV3-11', 'IGKV1-12', 'IGKV1-13', 'IGKV3-15', 'IGKV1-16', 'IGKV1-17', 'IGKV2-18', 'IGKV3-20', 'IGKV6-21', 'IGKV2-24', 'IGKV2-26', 'IGKV1-27', 'IGKV2-28', 'IGKV2-29', 'IGKV2-30', 'IGKV1-33', 'IGKV1-37', 'IGKV1-39', 'IGKV2-40', 'IGKV2D-40', 'IGKV1D-39', 'IGKV1D-37', 'IGKV1D-33', 'IGKV2D-30', 'IGKV2D-29', 'IGKV2D-28', 'IGKV1D-27', 'IGKV2D-26', 'IGKV2D-24', 'IGKV6D-21', 'IGKV3D-20', 'IGKV2D-18', 'IGKV6D-41', 'IGKV1D-17', 'IGKV1D-16', 'IGKV3D-15', 'IGKV1D-13', 'IGKV1D-12', 'IGKV3D-11', 'IGKV1D-42', 'IGKV1D-43', 'IGKV1D-8', 'IGKV3D-7', 'IGKV2OR2-1', 'IGKV2OR2-2', 'IGKV2OR2-10', 'IGKV1OR2-6', 'IGKV1OR2-108', 'IGHEP2', 'IGKV1OR-2', 'IGHMBP2', 'IGHE', 'IGHGP', 'IGHJ6', 'IGHJ5', 'IGHJ4', 'IGHJ3', 'IGHJ2', 'IGHJ1', 'IGHV6-1', 'I

In [8]:
# Cell 7: BROAD SCAN — ALL genes in C5 148-panel for PlasmaB/B
# Pattern: chronic common (IT+IA+CR vs AR)

# Load C5 gene list if available
c5_path = os.path.join(RESULTS_DIR, 'C5')
c5_genes = []
if os.path.exists(c5_path):
    for f in os.listdir(c5_path):
        if f.endswith('.csv'):
            df_tmp = pd.read_csv(os.path.join(c5_path, f))
            for col in df_tmp.columns:
                genes = df_tmp[col].dropna().tolist()
                c5_genes.extend([g for g in genes if g in adata.var_names])

# Also try C3 gene list
c3_path = os.path.join(RESULTS_DIR, 'C3_gene_list_196genes.csv')
if os.path.exists(c3_path):
    df_c3 = pd.read_csv(c3_path)
    for col in df_c3.columns:
        c5_genes.extend([g for g in df_c3[col].dropna().tolist() if g in adata.var_names])

# Deduplicate
c5_genes = sorted(set(c5_genes))
print(f'Total unique genes from C3/C5: {len(c5_genes)}')

# If no gene lists found, use the genes already in gene_expr
if not c5_genes:
    c5_genes = sorted(gene_expr.keys())
    print(f'Using pre-extracted genes: {len(c5_genes)}')

# Pre-extract any missing genes
for gene in c5_genes:
    if gene not in gene_expr:
        try:
            col = adata[:, gene].X
            if hasattr(col, 'toarray'):
                col = col.toarray().flatten()
            else:
                col = np.asarray(col).flatten()
            gene_expr[gene] = col
        except:
            pass

print(f'Total genes in gene_expr: {len(gene_expr)}')

# Broad scan: PlasmaB + B, all genes
print(f'\nScanning {len(c5_genes)} genes × 2 lineages × 2 tissues...')

broad_chronic = []

for gene in c5_genes:
    if gene not in gene_expr:
        continue
    for lin in ['PlasmaB', 'B']:
        for tis in ['Liver', 'Blood']:
            nl_it = fast_donor_test(gene, lin, tis, 'NL', 'IT')
            nl_ia = fast_donor_test(gene, lin, tis, 'NL', 'IA')
            nl_cr = fast_donor_test(gene, lin, tis, 'NL', 'CR')
            nl_ar = fast_donor_test(gene, lin, tis, 'NL', 'AR')

            it_sig = nl_it and nl_it['p'] < 0.05
            ia_sig = nl_ia and nl_ia['p'] < 0.05
            cr_sig = nl_cr and nl_cr['p'] < 0.05
            ar_sig = nl_ar and nl_ar['p'] < 0.05

            chronic_count = sum([it_sig, ia_sig, cr_sig])

            # Chronic common: >=2 chronic sig, AR NOT sig
            if chronic_count >= 2 and not ar_sig:
                # Check direction consistency
                dirs = []
                for r in [nl_it, nl_ia, nl_cr]:
                    if r and r['p'] < 0.05:
                        dirs.append(r['dir'])
                consistent_dir = len(set(dirs)) == 1

                broad_chronic.append({
                    'Gene': gene, 'Lineage': lin, 'Tissue': tis,
                    'NL_IT': fmt(nl_it), 'NL_IA': fmt(nl_ia),
                    'NL_CR': fmt(nl_cr), 'NL_AR': fmt(nl_ar),
                    'chronic_n': chronic_count,
                    'direction': dirs[0] if dirs else '?',
                    'consistent': consistent_dir,
                })

print(f'\nChronic common in PlasmaB/B: {len(broad_chronic)}')

# Sort by gene and display
print(f'\n{"Gene":>10s} | {"Lineage":>8s} | {"Tissue":>6s} | {"Dir":>3s} | '
      f'{"NL->IT":>18s} | {"NL->IA":>18s} | {"NL->CR":>18s} | {"NL->AR":>18s}')
print('-'*120)
for r in sorted(broad_chronic, key=lambda x: (x['Gene'], x['Lineage'], x['Tissue'])):
    print(f'{r["Gene"]:>10s} | {r["Lineage"]:>8s} | {r["Tissue"]:>6s} | '
          f'{r["direction"]:>3s} | '
          f'{r["NL_IT"]:>18s} | {r["NL_IA"]:>18s} | '
          f'{r["NL_CR"]:>18s} | {r["NL_AR"]:>18s}')

Total unique genes from C3/C5: 149
Total genes in gene_expr: 388

Scanning 149 genes × 2 lineages × 2 tissues...

Chronic common in PlasmaB/B: 26

      Gene |  Lineage | Tissue | Dir |             NL->IT |             NL->IA |             NL->CR |             NL->AR
------------------------------------------------------------------------------------------------------------------------
        AR |  PlasmaB |  Blood |   + | *from_zero p=0.007 | *from_zero p=0.042 |  from_zero p=0.302 | (t)from_zero p=0.079
   ATP5F1B |  PlasmaB |  Blood |   + |      *+25% p=0.016 |       +27% p=0.286 |      *+22% p=0.036 |    (t)+28% p=0.071
      BCL2 |        B |  Blood |   + |      *+84% p=0.008 |    (t)+85% p=0.064 |     *+161% p=0.036 |      +122% p=0.143
      BCL6 |        B |  Blood |   + |     *+218% p=0.012 |     *+402% p=0.019 |       +59% p=0.764 |       +64% p=0.549
     CASP8 |  PlasmaB |  Liver |   + |     *+220% p=0.022 |     *+208% p=0.022 |      +158% p=0.243 |       +41% p=0.697
    

In [9]:
# Cell 8: Donor-level values for key findings

STAGE_ORDER = ['NL', 'IT', 'IA', 'AR', 'CR']

# Key genes to show donor values
key_combos = [
    ('PRDM1', 'PlasmaB', 'Liver'),
    ('PRDM1', 'PlasmaB', 'Blood'),
    ('PRDM1', 'B', 'Liver'),
    ('PRDM1', 'B', 'Blood'),
    ('IRF4', 'PlasmaB', 'Liver'),
    ('IRF4', 'PlasmaB', 'Blood'),
    ('IRF4', 'B', 'Liver'),
    ('JCHAIN', 'PlasmaB', 'Liver'),
    ('JCHAIN', 'PlasmaB', 'Blood'),
    ('JCHAIN', 'B', 'Liver'),
    ('XBP1', 'PlasmaB', 'Liver'),
    ('XBP1', 'PlasmaB', 'Blood'),
    ('TYROBP', 'PlasmaB', 'Liver'),
    ('TYROBP', 'PlasmaB', 'Blood'),
    ('FCER1G', 'PlasmaB', 'Liver'),
    ('FCER1G', 'PlasmaB', 'Blood'),
    ('GZMB', 'PlasmaB', 'Liver'),
    ('DNMT1', 'PlasmaB', 'Liver'),
    ('DNMT3A', 'PlasmaB', 'Liver'),
]

print('=== DONOR-LEVEL VALUES FOR KEY PlasmaB/B GENES ===')
for gene, lin, tis in key_combos:
    if gene not in gene_expr:
        continue
    mask = (obs[COL_LINEAGE] == lin) & (obs[COL_TISSUE] == tis)
    cells = obs[mask].copy()
    cell_indices = np.where(mask.values)[0]
    if len(cell_indices) == 0:
        continue
    cells['expr'] = gene_expr[gene][cell_indices]
    dm = cells.groupby([COL_DONOR, COL_STAGE])['expr'].mean().reset_index()

    parts = []
    for stage in STAGE_ORDER:
        vals = dm[dm[COL_STAGE] == stage]['expr'].dropna().values
        if len(vals) > 0:
            parts.append(f'{stage}(n={len(vals)})={np.mean(vals):.4f}')

    if parts:
        print(f'{gene:>10s} {lin:>8s} {tis:>6s}: {" | ".join(parts)}')

=== DONOR-LEVEL VALUES FOR KEY PlasmaB/B GENES ===
     PRDM1  PlasmaB  Liver: NL(n=6)=0.6457 | IT(n=5)=0.3734 | IA(n=5)=0.2942 | AR(n=3)=0.2768 | CR(n=3)=0.2687
     PRDM1  PlasmaB  Blood: NL(n=5)=0.5613 | IT(n=5)=0.3513 | IA(n=4)=0.2926 | AR(n=3)=0.6006 | CR(n=3)=0.4439
     PRDM1        B  Liver: NL(n=6)=0.0248 | IT(n=5)=0.0144 | IA(n=5)=0.0167 | AR(n=3)=0.0202 | CR(n=3)=0.0100
     PRDM1        B  Blood: NL(n=5)=0.0807 | IT(n=5)=0.0154 | IA(n=4)=0.0170 | AR(n=3)=0.0133 | CR(n=3)=0.0157
      IRF4  PlasmaB  Liver: NL(n=6)=0.5887 | IT(n=5)=1.4694 | IA(n=5)=1.4588 | AR(n=3)=1.1415 | CR(n=3)=1.0286
      IRF4  PlasmaB  Blood: NL(n=5)=0.6991 | IT(n=5)=0.8376 | IA(n=4)=0.7451 | AR(n=3)=0.9204 | CR(n=3)=0.8747
      IRF4        B  Liver: NL(n=6)=0.2217 | IT(n=5)=0.2195 | IA(n=5)=0.1770 | AR(n=3)=0.1915 | CR(n=3)=0.2153
    JCHAIN  PlasmaB  Liver: NL(n=6)=4.1488 | IT(n=5)=3.0469 | IA(n=5)=3.1198 | AR(n=3)=3.9044 | CR(n=3)=3.5330
    JCHAIN  PlasmaB  Blood: NL(n=5)=4.8710 | IT(n=5)=4.2979 |

In [10]:
# Cell 9: SYNTHESIS — Categorize PlasmaB reprogramming evidence

print('='*80)
print('SYNTHESIS: PlasmaB REPROGRAMMING IN CHRONIC HBV')
print('='*80)

print()
print('=== Category 1: Antibody production genes (SUPPRESSED in chronic?) ===')
for gene in ['PRDM1', 'XBP1', 'JCHAIN', 'MZB1', 'IRF4']:
    if gene not in gene_expr:
        print(f'  {gene}: NOT IN DATASET')
        continue
    for lin in ['PlasmaB', 'B']:
        for tis in ['Liver', 'Blood']:
            nl_it = fast_donor_test(gene, lin, tis, 'NL', 'IT')
            nl_ar = fast_donor_test(gene, lin, tis, 'NL', 'AR')
            if nl_it or nl_ar:
                print(f'  {gene:>10s} {lin:>8s} {tis:>6s}: '
                      f'NL->IT={fmt(nl_it):>18s}  NL->AR={fmt(nl_ar):>18s}')

print()
print('=== Category 2: Cytotoxic program (GAINED in chronic?) ===')
for gene in ['TYROBP', 'FCER1G', 'GZMB', 'GNLY']:
    for lin in ['PlasmaB', 'B']:
        for tis in ['Liver', 'Blood']:
            nl_it = fast_donor_test(gene, lin, tis, 'NL', 'IT')
            nl_ar = fast_donor_test(gene, lin, tis, 'NL', 'AR')
            if nl_it and nl_it['p'] < 0.10:
                print(f'  {gene:>10s} {lin:>8s} {tis:>6s}: '
                      f'NL->IT={fmt(nl_it):>18s}  NL->AR={fmt(nl_ar):>18s}')

print()
print('=== Category 3: Epigenetic modification (in chronic?) ===')
for gene in ['DNMT1', 'DNMT3A', 'TET2']:
    for lin in ['PlasmaB', 'B']:
        for tis in ['Liver', 'Blood']:
            nl_it = fast_donor_test(gene, lin, tis, 'NL', 'IT')
            nl_ar = fast_donor_test(gene, lin, tis, 'NL', 'AR')
            if nl_it and nl_it['p'] < 0.10:
                print(f'  {gene:>10s} {lin:>8s} {tis:>6s}: '
                      f'NL->IT={fmt(nl_it):>18s}  NL->AR={fmt(nl_ar):>18s}')

SYNTHESIS: PlasmaB REPROGRAMMING IN CHRONIC HBV

=== Category 1: Antibody production genes (SUPPRESSED in chronic?) ===
       PRDM1  PlasmaB  Liver: NL->IT=      -42% p=0.247  NL->AR=      -57% p=0.262
       PRDM1  PlasmaB  Blood: NL->IT=      -37% p=0.151  NL->AR=       +7% p=1.000
       PRDM1        B  Liver: NL->IT=      -42% p=0.779  NL->AR=      -19% p=0.897
       PRDM1        B  Blood: NL->IT=      -81% p=0.675  NL->AR=      -84% p=0.764
        XBP1  PlasmaB  Liver: NL->IT=       -5% p=0.931  NL->AR=      -11% p=0.381
        XBP1  PlasmaB  Blood: NL->IT=      -15% p=0.309  NL->AR=       +2% p=1.000
        XBP1        B  Liver: NL->IT=      +24% p=0.247  NL->AR=      -28% p=0.381
        XBP1        B  Blood: NL->IT=      +62% p=0.151  NL->AR=   (t)+58% p=0.071
      JCHAIN  PlasmaB  Liver: NL->IT=     *-27% p=0.017  NL->AR=       -6% p=0.714
      JCHAIN  PlasmaB  Blood: NL->IT=   (t)-12% p=0.095  NL->AR=       +1% p=0.786
      JCHAIN        B  Liver: NL->IT=   (t)+56% p=

In [11]:
# Cell 10: Save results

# Save chronic common findings
if broad_chronic:
    df_chronic = pd.DataFrame(broad_chronic)
    path1 = os.path.join(SAVE_DIR, 'PlasmaB_B_chronic_common.csv')
    df_chronic.to_csv(path1, index=False, encoding='utf-8-sig')
    print(f'Saved: {path1}')

# Save full scan
all_rows = []
all_genes_scan = sorted(gene_expr.keys())
for gene in all_genes_scan:
    for lin in ['PlasmaB', 'B']:
        for tis in ['Liver', 'Blood']:
            nl_it = fast_donor_test(gene, lin, tis, 'NL', 'IT')
            nl_ia = fast_donor_test(gene, lin, tis, 'NL', 'IA')
            nl_cr = fast_donor_test(gene, lin, tis, 'NL', 'CR')
            nl_ar = fast_donor_test(gene, lin, tis, 'NL', 'AR')
            any_sig = any(r and r['p'] < 0.05 for r in [nl_it, nl_ia, nl_cr, nl_ar])
            if any_sig:
                all_rows.append({
                    'Gene': gene, 'Lineage': lin, 'Tissue': tis,
                    'NL->IT': fmt(nl_it), 'NL->IA': fmt(nl_ia),
                    'NL->CR': fmt(nl_cr), 'NL->AR': fmt(nl_ar),
                })

df_all = pd.DataFrame(all_rows)
path2 = os.path.join(SAVE_DIR, 'PlasmaB_B_full_scan.csv')
df_all.to_csv(path2, index=False, encoding='utf-8-sig')
print(f'Saved: {path2} ({len(df_all)} rows)')

print(f'\n=== MANUSCRIPT DECISION ===')
print('If PRDM1/XBP1/JCHAIN suppressed in IT-IA-CR but not AR:')
print('  → New section: "PlasmaB Functional Reprogramming"')
print('  → Direct evidence: antibody production machinery blocked')
print('  → Combined with TYROBP/FCER1G/GZMB gain: dual reprogramming')
print('  → Clinical: explains anti-HBs failure in chronic infection')
print('\n✅ Investigation complete.')

Saved: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/PlasmaB_antibody_investigation_26Mar13/PlasmaB_B_chronic_common.csv
Saved: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/PlasmaB_antibody_investigation_26Mar13/PlasmaB_B_full_scan.csv (234 rows)

=== MANUSCRIPT DECISION ===
If PRDM1/XBP1/JCHAIN suppressed in IT-IA-CR but not AR:
  → New section: "PlasmaB Functional Reprogramming"
  → Direct evidence: antibody production machinery blocked
  → Combined with TYROBP/FCER1G/GZMB gain: dual reprogramming
  → Clinical: explains anti-HBs failure in chronic infection

✅ Investigation complete.
